In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

laps = pd.read_pickle("../data/prepared_race_laps_2022_2025.pkl")

keys = ["Season", "RoundNumber", "DriverNumber"]

usable = (
    laps["LapTimeSeconds"].notna()
    & laps["IsPitLap"].eq(False)
    & laps["TrackStatus"].eq("1")
    & laps["IsAccurate"].eq(True)
)

sequences = (
    laps.loc[usable]
    .sort_values(keys + ["LapNumber"])
    .copy()
)

sequences["Segment"] = (
    sequences.groupby(keys)["LapNumber"]
    .diff().ne(1).cumsum()
)


def predict_arima(history):
    baseline = history[-1]

    if len(history) < 10:
        return baseline, "Baseline: short history"

    try:
        fitted = ARIMA(
            history,
            order=(0, 1, 1),
            trend="n"
        ).fit()

        prediction = fitted.forecast(steps=1)[0]

        if (
            fitted.mle_retvals.get("converged", False)
            and np.isfinite(prediction)
        ):
            return prediction, "ARIMA"

    except (ValueError, np.linalg.LinAlgError):
        return baseline, "Baseline: fit failed"

    return baseline, "Baseline: unreliable fit"


records = []
segments = sequences.groupby("Segment")

for number, (_, sequence) in enumerate(segments, start=1):
    times = sequence["LapTimeSeconds"].to_numpy()

    smoothing = (
        sequence["LapTimeSeconds"]
        .ewm(alpha=0.9, adjust=False)
        .mean()
        .to_numpy()
    )

    for i, (_, row) in enumerate(sequence.iterrows()):
        if not row["EligiblePair"]:
            continue

        prediction, method = predict_arima(times[:i + 1])

        records.append({
            "Season": row["Season"],
            "RoundNumber": row["RoundNumber"],
            "DriverNumber": row["DriverNumber"],
            "PredictedLap": int(row["NextLapNumber"]),
            "HistoryLaps": i + 1,
            "Baseline": times[i],
            "Smoothing": smoothing[i],
            "ARIMA": prediction,
            "ARIMAMethod": method
        })

    if number % 200 == 0:
        print(f"Processed {number} / {segments.ngroups} sequences")

predictions = pd.DataFrame(records)

app_data = laps[
    [
        "Season",
        "RoundNumber",
        "EventName",
        "Driver",
        "DriverNumber",
        "LapNumber",
        "LapTimeSeconds"
    ]
].rename(columns={
    "LapNumber": "PredictedLap",
    "LapTimeSeconds": "Actual"
})

app_data = app_data.merge(
    predictions,
    on=keys + ["PredictedLap"],
    how="left",
    validate="one_to_one"
)

app_data["PredictionAvailable"] = app_data["Smoothing"].notna()

app_data["ARIMAMethod"] = app_data["ARIMAMethod"].fillna(
    "Unavailable: outside evaluated scope"
)

app_data["DatasetSplit"] = app_data["Season"].map({
    2022: "Training",
    2023: "Training",
    2024: "Validation",
    2025: "Test"
})

app_data = app_data.sort_values(
    keys + ["PredictedLap"]
).reset_index(drop=True)

available = app_data.loc[app_data["PredictionAvailable"]]

assert len(available) == int(laps["EligiblePair"].sum())
assert np.isfinite(
    available[["Actual", "Baseline", "Smoothing", "ARIMA"]].to_numpy()
).all()

app_data.to_pickle("../data/app_predictions.pkl")
app_data.to_csv("../data/app_predictions.csv", index=False)

print("Recorded laps:", len(app_data))
print("Laps with predictions:", len(available))

display(
    app_data.loc[
        app_data["Season"].eq(2024)
        & app_data["EventName"].eq("Belgian Grand Prix")
        & app_data["Driver"].eq("LEC")
        & app_data["PredictedLap"].eq(21)
    ]
)

c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Che

Processed 200 / 6584 sequences


c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros

Processed 400 / 6584 sequences


c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting paramete

Processed 600 / 6584 sequences


... (57 outputs left out)

Processed 6400 / 6584 sequences


c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\Francisco\Desktop\Work\Semester 3\Practice Challenge\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = 

Recorded laps: 101290
Laps with predictions: 78236


,Season,RoundNumber,EventName,Driver,DriverNumber,PredictedLap,Actual,HistoryLaps,Baseline,Smoothing,ARIMA,ARIMAMethod,PredictionAvailable,DatasetSplit
63048,2024,14,Belgian Grand Prix,LEC,16,21.0,108.38,5.0,108.279,108.28196,108.279,Baseline: short history,True,Validation
